# 10 — The library in one tour: C#
PlotSharp source-build charts: Release build of MathemaSharp and ../PlotSharp/src/PlotSharp/PlotSharp/PlotSharp.csproj. Runner: dotnet run --file ops/RunMigratedCoreNotebook.cs -- --id tour. Five SVGs verified and Edge screenshots observed; the C#/PySharp calculations and numerical comparison were not changed by the graphics migration.

## Usage and verification status
The tour uses the generic C# APIs; it does not automatically prove agreement with the PySharp twin. The kernels have different per-format coverage: consult the inventory and the Reference. Lanczos special functions do not promise 34 Decimal128 digits; simulation by default starts from double; GARCH remains a grid estimator. This tour does not load the NDSharp bridge.
[API catalogue](12-api-reference.ipynb), [consolidation](11-consolidation.ipynb), [limits and coverage](../../ARCHITECTURE.md).
The same sequence in **C#** and in **PySharp**, the same data (same seed) and the same numbers: every line `label: values` printed here appears identical in the other notebook, and the `TourNotebooksAgreeAcrossLanguages` test verifies it. In C# the **generic** API is used (`T = double`); in PySharp the `MathemaSharp.TimeSeries.Interop` facade, because its reflection neither passes `ReadOnlySpan<T>` nor resolves generic methods.

**JupyterSharp** kernel (C#) on the **.NET 11** runtime (`scripts/run-notebooks.ps1`). Part II of the book: [the library](../../refs/timeseries/01-libreria/README.md).

In [ ]:
#load "../shared/MathemaSharp.PlotSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });

// Simulated signal: AR(2) with φ = (0.6, −0.2), σ = 1, 600 samples. Last 20 = test.
var y = Simulation.Arma<double>(new Random(2026), 600, 0, new[] { 0.6, -0.2 }, Array.Empty<double>(), 1);
double[] train = y[..580], test = y[580..];
P("n", y.Length);

n: 600.000000

## 1 · Basic statistics (ch. 2)

In [ ]:
P("mean", SeriesStatistics.Mean<double>(train));
P("acf", SeriesStatistics.Acf<double>(train, 5));
P("pacf", SeriesStatistics.Pacf<double>(train, 5));
var lb = SeriesStatistics.LjungBox<double>(train, 10, 0);
P("ljungbox", new[] { lb.Statistic, lb.PValue });
P("quantiles", new[] { SeriesStatistics.Quantile<double>(train, 0.1), SeriesStatistics.Quantile<double>(train, 0.5), SeriesStatistics.Quantile<double>(train, 0.9) });
Chart.Stems("PACF: cuts off after lag 2", SeriesStatistics.Pacf<double>(train, 12), 1.96 / Math.Sqrt(train.Length));

mean: 0.215617
acf: 1.000000 0.499273 0.098564 -0.079481 -0.020036 0.027441
pacf: 0.499273 -0.200752 -0.053216 0.097401 -0.014759
ljungbox: 156.235427 0.000000
quantiles: -1.348212 0.249128 1.754431

PACF: cuts off after lag 2 -0.4 -0.2 0 0.2 0.4 0.6 2.5 5 7.5 10 12.5 Lag PACF: cuts off after lag 2 · x=1: 0.499 PACF: cuts off after lag 2 · x=2: -0.201 PACF: cuts off after lag 2 · x=3: -0.053 PACF: cuts off after lag 2 · x=4: 0.097 PACF: cuts off after lag 2 · x=5: -0.015 PACF: cuts off after lag 2 · x=6: -0.039 PACF: cuts off after lag 2 · x=7: -0.002 PACF: cuts off after lag 2 · x=8: 0.017 PACF: cuts off after lag 2 · x=9: -0.039 PACF: cuts off after lag 2 · x=10: 0.018 PACF: cuts off after lag 2 · x=11: 0.078 PACF: cuts off after lag 2 · x=12: -0.059 +band · x=0.5: 0.081 +band · x=12.5: 0.081 -band · x=0.5: -0.081 -band · x=12.5: -0.081 -band +band PACF: cuts off after lag 2

## 2 · Reference forecasts and metrics (ch. 3)

In [ ]:
foreach (var (name, f) in new (string, double[])[] {
    ("mean", ReferenceForecasts.Mean<double>(train, 20)),
    ("naive", ReferenceForecasts.Naive<double>(train, 20)),
    ("drift", ReferenceForecasts.Drift<double>(train, 20)) })
    P($"mase_{name}", Accuracy.Mase<double>(test, f, train, 1));

mase_mean: 0.766674
mase_naive: 0.798872
mase_drift: 0.805145

## 3 · AR and ARIMA (ch. 4)

In [ ]:
var ar = ArModel.FitYuleWalker<double>(train, 2);
P("ar_phi", ar.Phi);
P("ar_sigma2", ar.InnovationVariance);
P("ar_forecast", ar.Forecast(train, 5));
var arma = ArimaModel.FitHannanRissanen<double>(train, 1, 0, 1);
P("arma_params", new[] { arma.Constant, arma.Phi[0], arma.Theta[0], arma.InnovationVariance });

ar_phi: 0.599503 -0.200752
ar_sigma2: 1.051243
ar_forecast: 0.565835 0.394854 0.252763 0.201904 0.199939
arma_params: 0.169949 0.265452 0.316239 1.071890

## 4 · Exponential smoothing (ch. 5)

In [ ]:
var alpha = ExponentialSmoothing.FitSesAlpha<double>(train);
P("ses_alpha", alpha);
P("ses_forecast", ExponentialSmoothing.Ses<double>(train, alpha, 3).Forecast);
P("holt_damped", ExponentialSmoothing.Holt<double>(train, 0.4, 0.2, 5, 0.9).Forecast);

// Series with trend and annual seasonality (period 12), on the same noise.
var seasonal = y.Select((v, t) => v + 0.02 * t + 3 * Math.Sin(2 * Math.PI * t / 12)).ToArray();
var hw = ExponentialSmoothing.HoltWinters<double>(seasonal[..588], 12, 0.3, 0.05, 0.2, 12, 1.0);
P("hw_forecast", hw.Forecast[..6]);
P("hw_mase", Accuracy.Mase<double>(seasonal[588..], hw.Forecast, seasonal[..588], 12));

ses_alpha: 0.697544
ses_forecast: 0.047987 0.047987 0.047987
holt_damped: -0.117570 -0.049685 0.011411 0.066398 0.115886
hw_forecast: 12.429914 13.911004 14.379396 16.006896 15.858115 14.173691
hw_mase: 0.668512

## 5 · Kalman filter (ch. 6)

In [ ]:
var kf = KalmanFilter.LocalLevel<double>(0.1, 1.0, train[0], 10.0);
var kr = kf.Filter(train[..100]);
P("kalman_gain_last", kr.FirstGain[^1]);
P("kalman_loglik", kr.LogLikelihood);
P("kalman_steady_gain", KalmanFilter.LocalLevelSteadyStateGain<double>(0.1, 1.0));

kalman_gain_last: 0.270156
kalman_loglik: -150.844287
kalman_steady_gain: 0.270156

## 6 · VAR (ch. 7)
Two coupled series: $x$ = the first 400 samples, $z_t=0.5\,z_{t-1}+0.3\,x_{t-1}+0.5\,e_t$.

In [ ]:
var noise = Enumerable.Range(0, 400).Select(i => Simulation.StandardNormal(new Random(7 + i))).ToArray();
var data = new double[400, 2];
for (var t = 0; t < 400; t++)
{
    data[t, 0] = y[t];
    data[t, 1] = t == 0 ? 0.5 * noise[t] : 0.5 * data[t - 1, 1] + 0.3 * y[t - 1] + 0.5 * noise[t];
}
var var1 = VarModel.Fit<double>(data, 1);
P("var_a1", new[] { var1.Coefficients[0][0, 0], var1.Coefficients[0][0, 1], var1.Coefficients[0][1, 0], var1.Coefficients[0][1, 1] });
P("var_stable", var1.IsStableOrder1Bivariate() ? 1 : 0);

var_a1: 0.562976 -0.294915 0.350940 0.240238
var_stable: 1.000000

## 7 · GARCH (ch. 8)

In [ ]:
var truth = new Garch11<double>(0.05, 0.1, 0.85);
var shocks = Simulation.Garch(new Random(7), truth, 1500);
var fit = Garch11.FitVarianceTargeting<double>(shocks);
P("garch_fit", new[] { fit.Omega, fit.Alpha, fit.Beta });
P("garch_forecast", fit.ForecastVariance(shocks, 5));

garch_fit: 0.062030 0.139250 0.792925
garch_forecast: 1.402143 1.369073 1.338245 1.309509 1.282721

## 8 · STL (ch. 9)

In [ ]:
var stl = Stl.Decompose<double>(seasonal, 12, 7, null, 2, 0);
P("stl_trend_ends", new[] { stl.Trend[0], stl.Trend[^1] });
P("stl_seasonal", stl.Seasonal[..12]);
Chart.Panels("STL of the series with trend and seasonality", new Line("observed", seasonal), new Line("trend", stl.Trend), new Line("seasonality", stl.Seasonal), new Line("remainder", stl.Remainder));

stl_trend_ends: 0.949002 11.739431
stl_seasonal: -0.594054 0.782731 2.468831 3.594041 2.850572 1.215369 0.382547 -2.463216 -3.794538 -3.227943 -1.416305 -0.491050

STL of the series with trend and seasonality observed -5 0 5 10 15 20 0 200 400 600 <path d='M64 114.66L65.05 108.82L66.11 103.98L67.16 100.21L68.21 102.73L69.27 106.67L70.32 107.14L71.37 117.16L72.43 122.86L73.48 119.24L74.53 115.19L75.59 116.25L76.64 114.99L77.69 112.42L78.75 106.38L79.8 102.65L80.85 105.2L81.91 112.08L82.96 116.84L84.01 124.18L85.07 128.78L86.12 129.48L87.17 124.26L88.23 117.1L89.28 107.77L90.33 105.93L91.39 100.47L92.44 99.16L93.49 100.42L94.55 106.6L95.6 114.07L96.65 121.93L97.71 119.66L98.76 118.35L99.81 115.32L100.87 109.1L101.92 108.9L102.97 103.51L104.03 103.27L105.08 99.28L106.13 101.04L107.19 99.99L108.24 106.24L109.29 111.77L110.35 122.13L111.4 119.42L112.45 121.51L113.51 119.49L114.56 107.71L115.61 107.19L116.67 99.46L117.72 96.66L118.77 98.02L119.83 102.26L120.88 105.2L121.93 109.58L122.99 112.54L124.04 111.71L125.09 118.32L126.15 118.85L127.2 112.73L128.25 102.41L129.31 98.1L130.36 95.55L131.41 95.65L132.47 103.49L133.52 104.14L134.57 114.78L135.63 115.55L136.68 112.75L137.73 117.89L138.79 117.09L139.84 111.28L140.89 107.08L141.95 99.06L143 103.49L144.05 109L145.11 108.17L146.16 112.69L147.21 120.04L148.27 119.54L149.32 121.88L150.37 113.7L151.43 113.34L152.48 112.94L153.53 109.37L154.59 105.29L155.64 103.88L156.69 98.62L157.75 106.87L158.8 111.49L159.85 115.96L160.91 117.4L161.96 119.59L163.01 115.02L164.07 105.12L165.12 101.27L166.17 100.93L167.23 97.83L168.28 90.72L169.33 92.6L170.39 98.97L171.44 109.13L172.49 118.2L173.55 114.65L174.6 113.57L175.65 115.91L176.71 111.69L177.76 102.14L178.81 96.99L179.87 91.82L180.92 98.57L181.97 103.02L183.03 104.34L184.08 105.46L185.13 107.36L186.19 107.06L187.24 112.8L188.29 118.82L189.35 112.78L190.4 104.31L191.45 97.36L192.51 88.12L193.56 88.25L194.61 81.93L195.67 89.1L196.72 99.88L197.77 111.93L198.83 116.62L199.88 113.38L200.93 110.7L201.99 105.93L203.04 101.46L204.09 102.71L205.15 101.45L206.2 95.66L207.25 97.98L208.31 107.6L209.36 104.95L210.41 111.73L211.47 118.72L212.52 116.88L213.57 109.61L214.63 104.93L215.68 104.43L216.73 96.12L217.79 92.58L218.84 93.3L219.89 99.19L220.95 101.46L222 103.03L223.05 111.85L224.11 113.23L225.16 116.73L226.21 108.83L227.27 100.69L228.32 97.82L229.37 96.28L230.43 92.79L231.48 93.67L232.53 95.54L233.59 99.78L234.64 111.34L235.69 113.65L236.75 115.98L237.8 111.59L238.85 110.28L239.91 104.38L240.96 99.71L242.01 92.11L243.07 87.08L244.12 91.81L245.17 100.24L246.23 99.45L247.28 96.45L248.33 103.81L249.39 107.49L250.44 116.54L251.49 109.84L252.55 101.27L253.6 93.86L254.65 92.78L255.71 92.92L256.76 95.91L257.81 97.7L258.87 91.38L259.92 94.78L260.97 98.76L262.03 109.28L263.08 118.76L264.13 111.57L265.19 106.29L266.24 97.81L267.29 100.62L268.35 89.71L269.4 86.35L270.45 90.45L271.51 95.6L272.56 99.16L273.61 101.93L274.67 114.6L275.72 119.15L276.77 109.49L277.83 102.11L278.88 95.4L279.93 86.16L280.99 86.29L282.04 86.5L283.09 87.97L284.15 88.71L285.2 92.76L286.25 105.72L287.31 108.66L288.36 111.99L289.41 105.25L290.47 108.05L291.52 102.34L292.57 92.99L293.63 88.17L294.68 91.53L295.73 90.93L296.79 92L297.84 97.2L298.89 109.24L299.95 113.76L301 107.97L302.05 108.37L303.11 101.25L304.16 92.05L305.21 86.94L306.27 83.74L307.32 89.52L308.37 93.41L309.43 97.77L310.48 101.09L311.53 99.64L312.59 105.54L313.64 105.88L314.69 106.59L315.75 99.37L316.8 96.34L317.85 91.38L318.91 88.95L319.96 89.53L321.01 93.03L322.07 92.19L323.12 94.53L324.17 97.81L325.23 102.09L326.28 110.75L327.33 110.55L328.39 103.06L329.44 102.27L330.49 89.33L331.55 88.56L332.6 85.29L333.65 91.33L334.71 92.47L335.76 98.28L336.81 102.81L337.87 107.35L338.92 98.46L339.97 107.4L341.03 107.3L342.08 102.95L343.13 93.45L344.19 79.13L345.24 86.01L346.29 89.03L347.35 89.06L348.4 90.68L349.45 96.88L350.51 102.49L351.56 106.04L352.61 104.31L353.67 97.59L354.72 97.1L355.77 90.61L356.83 82.15L357.88 82.54L358.93 86.34L359.99 89.31L361.04 98.19L362.09 103.6L363.15 106.2L364.2 107.56L365.25 95.24L366.31 92.15L367.36 91.05L368.41 85.74L369.47 83.07L370.52 85.89L371.57 87.65L372.63 95.67L3

## 9 · C# only: the same algorithm in other formats
Levinson-Durbin on the same series in `float`, `double` and `Decimal128`. The difference between the results is only arithmetic (same noise): the PACF at lag 1 changes from the seventh digit in `float` to the sixteenth in `double`.

In [ ]:
using System.Numerics;
foreach (var (name, value) in new (string, double)[]
{
    ("float", double.CreateChecked(SeriesStatistics.Pacf<float>(train.Select(float.CreateChecked).ToArray(), 1)[0])),
    ("double", SeriesStatistics.Pacf<double>(train, 1)[0]),
    ("decimal128", double.CreateChecked(SeriesStatistics.Pacf<Decimal128>(train.Select(Decimal128.CreateChecked).ToArray(), 1)[0])),
})
    Console.WriteLine($"pacf1_{name}: {value:R}");

pacf1_float: 0.4992733895778656
pacf1_double: 0.49927325354040686
pacf1_decimal128: 0.49927325354040664